# 03 · FlashAttention —— 流水线小车，一次拉一车

**家族位置**：08 生产级优化第 3 站。02 的 Cache 省的是重复算，本章省的是搬运：标准注意力把 S×S 中间矩阵全物化在 HBM，Flash 分块算、S×S 永不落地。

**学习目标**：理解 IO 才是瓶颈（不是 FLOPs）；online softmax 分块归约；数学等价验证；HBM 访问量公式；CPU 上计时口径诚实声明。

## 1. 原理：仓库一次拉一车

### 通俗理解

**一句话**：标准注意力像一次拉一仓库（S×S 全算出来堆显存）；Flash 像流水线小车，一次拉一块，在 SRAM 里算完只把结果堆回去，中间大矩阵从没落地。

### 结构账

```
标准： S=softmax(QK^T/√d)V，S×S 物化 → HBM 访问 O(S²)
Flash： 分块 online softmax：m_new=max(m, rowmax)，l=l·α+Σβ，acc=acc·α+βV
等价： 同一 softmax，分块算 → 误差仅 float 舍入（~1e-7）
本章： torch 无 SDPA，手写分块模拟算法正确性 + HBM 计数器；计时只作等价性旁证
```

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.models import naive_attention, flash_attention, naive_hbm
from common.utils import set_seed,setup_chinese_font
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__,'| SDPA:',hasattr(torch.nn.functional,'scaled_dot_product_attention'))
B,H,D=2,4,16
for S in [64,128,256,512]:
    torch.manual_seed(0)
    q,k,v=torch.randn(B,H,S,D),torch.randn(B,H,S,D),torch.randn(B,H,S,D)
    o1,_=naive_attention(q,k,v); o2,_=flash_attention(q,k,v,block=64)
    print(f'S={S} max|Δ|={(o1-o2).abs().max().item():.2e}',flush=True)

## 2. 等价性：分块算，数不变

In [ ]:
errs=[]
for S in [32,64,128,256,512]:
    torch.manual_seed(0)
    q,k,v=torch.randn(B,H,S,D),torch.randn(B,H,S,D),torch.randn(B,H,S,D)
    o1,_=naive_attention(q,k,v); o2,_=flash_attention(q,k,v,block=64)
    errs.append((o1-o2).abs().max().item())
    print(f'S={S} err={errs[-1]:.2e}',flush=True)
fig,ax=plt.subplots(figsize=(6,3.2))
ax.semilogx([32,64,128,256,512],errs,marker='o',color='#4C72B0')
ax.axhline(1e-5,color='red',ls='--',label='容差 1e-5')
ax.set_xlabel('S'); ax.set_ylabel('max|Δ|'); ax.legend()
ax.set_title('数学等价：误差仅 float 舍入，与 S 无关')
plt.tight_layout(); plt.savefig(FIGS/'fig1_equiv.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. IO：HBM 访问量公式对比

In [ ]:
Ss=[64,128,256,512,1024]; NB=64
na=[naive_hbm(S,D) for S in Ss]
fl=[]
for S in Ss:
    torch.manual_seed(0)
    q,k,v=torch.randn(1,1,S,D),torch.randn(1,1,S,D),torch.randn(1,1,S,D)
    _,h=flash_attention(q,k,v,block=NB); fl.append(h)
for S,a,b in zip(Ss,na,fl): print(f'S={S} naive={a} flash={b} ratio ×{a/b:.2f}',flush=True)
fig,ax=plt.subplots(figsize=(6,3.4))
ax.loglog(Ss,na,marker='o',label='naive O(S²)',color='#DD8452')
ax.loglog(Ss,fl,marker='o',label='flash O(S²/B+B·S)',color='#4C72B0')
ax.set_xlabel('S'); ax.set_ylabel('HBM elements'); ax.legend()
ax.set_title('IO 复杂度：S×S 物化 vs 分块流式')
plt.tight_layout(); plt.savefig(FIGS/'fig2_io.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. CPU 计时口径声明 + 块大小影响

In [ ]:
torch.manual_seed(0); q,k,v=torch.randn(B,H,256,D),torch.randn(B,H,256,D),torch.randn(B,H,256,D)
t0=time.perf_counter(); naive_attention(q,k,v); t1=time.perf_counter()
flash_attention(q,k,v); t2=time.perf_counter()
print(f'CPU S=256 naive {1000*(t1-t0):.1f}ms flash-py {1000*(t2-t1):.1f}ms（Python 循环开销大，只证等价不证加速）')
fig,ax=plt.subplots(figsize=(6,2.6)); ax.axis('off')
ax.text(0.02,0.7,'CPU 上手写分块更慢是预期的：真 Flash 是 CUDA kernel + SRAM，Python 循环只是算法模拟',fontsize=10)
ax.text(0.02,0.4,'生产加速比看 HBM 公式（fig2）+ GPU 实测，不看本机秒表',fontsize=10)
ax.text(0.02,0.1,'与 08-02 Cache 正交：Cache 省重算，Flash 省搬运，可叠加',fontsize=10,color='#1a6b3c')
ax.set_title('口径声明（诚实记录）')
plt.tight_layout(); plt.savefig(FIGS/'fig3_time.png',dpi=150,bbox_inches='tight'); plt.show()
bs=[16,32,64,128]; hs=[]
for b in bs:
    torch.manual_seed(0)
    qq,kk,vv=torch.randn(1,1,256,D),torch.randn(1,1,256,D),torch.randn(1,1,256,D)
    _,h=flash_attention(qq,kk,vv,block=b); hs.append(h)
fig,ax=plt.subplots(figsize=(6,3.2))
ax.plot(bs,hs,marker='o',color='#4C72B0')
ax.set_xlabel('block'); ax.set_ylabel('HBM elements')
ax.set_title('块大小：太大 SRAM 装不下，太小归约开销大（toy 示意）')
plt.tight_layout(); plt.savefig(FIGS/'fig4_block.png',dpi=150,bbox_inches='tight'); plt.show()
print('block→HBM:',list(zip(bs,hs)))

## 5. 总结与下一步

Flash 闭环：online softmax 分块归约 → 数学等价（~1e-7）→ HBM 公式对比 → CPU 口径声明。下一步 `04_LoRA_Finetune`：冻结大模型只训小矩阵（手写 LoRA，零新包）。